In [53]:
import numpy as np

class Board:

    def __init__(self):
        self.board = np.full((6, 7), ' ', dtype=str)
        self.empty = 42
        self.record = dict()
        self.record[repr(self)] = 1
        self.state = ' '

    def isLegal(self, isX: bool, move: tuple[int, int]):
        r, c = move[0], move[1] - 1
        if r == 0:
            return ((isX and self.board[5, c] == 'X') or (not isX and self.board[5, c] == 'O'))
        if r == 1:
            return self.board[6 - r, c] == ' '
        return (self.board[7 - r, c] != ' ' and self.board[6 - r, c] == ' ')
    
    def playMove(self, icon: str, move: tuple[int, int]):
        r, c = move[0], move[1] - 1
        if r == 0:
            self.board[1:, c] = self.board[:-1, c]
            self.board[0, c] = ' '
            self.empty += 1
        else:
            self.board[6 - r, c] = icon
            self.empty -= 1
        current_state = repr(self)
        if current_state in self.record:
            if self.record[current_state] == 2:
                self.state = "Game ends on a tie!"
            else:
                self.record[current_state] += 1
        else:
            self.record[current_state] = 1  
        self.checkWin('O' if icon == 'X' else 'X')
        self.checkWin(icon)

        if self.empty == 0 and self.state == ' ':
            self.state = "Game ends on a tie!"

    def checkWin(self, icon: str):
        b = (self.board == icon)
        if np.any(b[:, :-3] & b[:, 1:-2] & b[:, 2:-1] & b[:, 3:]):
            self.state = f"Game ends on {icon}'s win!"
            return
        if np.any(b[:-3, :] & b[1:-2, :] & b[2:-1, :] & b[3:, :]):
            self.state = f"Game ends on {icon}'s win!"
            return
        if np.any(b[:-3, :-3] & b[1:-2, 1:-2] & b[2:-1, 2:-1] & b[3:, 3:]):
            self.state = f"Game ends on {icon}'s win!"
            return
        if np.any(b[3:, :-3] & b[2:-1, 1:-2] & b[1:-2, 2:-1] & b[:-3, 3:]):
            self.state = f"Game ends on {icon}'s win!"
            return

    def __str__(self):
        printer = "  -----------------------------\n"
        for i in range(6):
            printer += f"{6 - i} |"
            for j in range(7):
                current = self.board[i, j]
                if current in ('X', 'O'):
                    printer += f" {current} |"
                else:
                    printer += "   |"
            printer += "\n  -----------------------------\n"
        printer += "    1   2   3   4   5   6   7"
        return printer
    
    def __repr__(self):
        return "".join(self.board.ravel())

In [54]:
class Player:
    def __init__(self, isX: bool):
        self.isX = isX

    def getPossibleMoves(self, board):
        if (board.empty == 0):
            return ["tie"]
        moves = []
        for i in range(1, 8):
            if (board.board[5][i - 1] == 'X' and self.isX) or (board.board[5][i - 1] == 'O' and not self.isX):
                moves.append((0, i))
            for j in range(1, 7):
                if board.board[6 - j][i - 1] == ' ':
                    moves.append((j, i))
                    break
        return moves

    def turn(self, board, printer=True):
        
        """ Essa função está aqui apenas de assinatura, 
            a sua implementação vai variar dependendo da subclasse que vem a seguir"""

        raise NotImplementedError("A subclasse deve implementar o método turn!")

    def __str__(self):
        return 'X' if self.isX else 'O'

In [55]:
class HumanPlayer(Player):
    def turn(self, board, printer=True):
        if printer:
            print(f"{self}'s turn")
            print(f"Possible moves: {self.getPossibleMoves(board)}")
            
        while True:
            toParse = input("Enter move coordinates separated by a comma: ").split(",")
            if (board.empty == 0 and len(toParse) == 0 and toParse[0] == "tie"):
                board.state = "Game ends on a tie!"
                break
            if len(toParse) != 2:
                print("Invalid move! Try again.")
                continue
                
            move = tuple((int(toParse[0]), int(toParse[1])))
            
            if move[0] not in range(0, 7) or move[1] not in range(1, 8):
                print("Invalid move! Try again.")
                continue
                
            if board.isLegal(self.isX, move):  
                board.playMove(str(self), move) 
                return move
            else:
                print("Invalid move! Try again.")

In [ ]:
print("--- Humano Vs Humano ---")
playerX = HumanPlayer(isX=True)
playerO = HumanPlayer(isX=False)
board = Board()

print(board)
while board.state == ' ':
    playerX.turn(board)
    print(board)
    if board.state != ' ':
        break
    playerO.turn(board)
    print(board)
print(board.state)

In [62]:
import copy
import math
import random 

class MCTSNode:

    def __init__(self, board, move=None, parent=None, constant=1.41, isX=True):
        self.board = copy.deepcopy(board)
        self.move = move
        self.parent = parent
        self.constant = constant
        self.isX = isX
        self.children = []
        self.wins = 0
        self.visits = 0
        self.untriedMoves = MCTSPlayer(isX).getPossibleMoves(self.board)

    def select(self):
        return max(self.children, key=lambda c: (c.wins / c.visits) + self.constant * np.sqrt(np.log(self.visits) / c.visits))
    
    def expand(self):
        move = self.untriedMoves.pop()
        next_board = copy.deepcopy(self.board)
        icon = 'X' if self.isX else 'O'
        next_board.playMove(icon, move)
        child_node = MCTSNode(next_board, move=move, parent=self, constant=self.constant, isX=not self.isX)
        self.children.append(child_node)
        return child_node
    
    def update(self, result):
        self.visits += 1
        if not self.isX: 
            self.wins += result
        else:
            self.wins += (1.0 - result)
        if self.parent:
            self.parent.update(result)

    def is_fully_expanded(self):
        return len(self.untriedMoves) == 0

    def is_terminal(self):
        return self.board.state != ' '

    def rollout(self):
        board_x, board_o = self.to_bitboard(self.board.board)
        current_x_turn = self.isX
        all_cells_mask = 0b111111_111111_111111_111111_111111_111111_111111
        while True:
            moves = self.get_bit_moves(board_x, board_o, current_x_turn)
            if not moves: return 0.5
            move_bit, is_pop = random.choice(moves)
            if is_pop:
                board_x, board_o = self.apply_bit_pop(board_x, board_o, move_bit)
            else:
                if current_x_turn: board_x |= move_bit
                else: board_o |= move_bit
            x_wins = self.check_bit_win(board_x)
            o_wins = self.check_bit_win(board_o)
            if x_wins and o_wins: return 0.5
            if x_wins: return 1.0
            if o_wins: return 0.0
            if (board_x | board_o) == all_cells_mask: return 0.5
            current_x_turn = not current_x_turn

    def to_bitboard(self, grid):
        board_x = 0
        board_o = 0
        for c in range(7):
            for r in range(6):
                shift = c * 7 + r
                if grid[5-r, c] == 'X':
                    board_x |= (1 << shift)
                elif grid[5-r, c] == 'O':
                    board_o |= (1 << shift)
        return board_x, board_o
    
    def get_bit_moves(self, board_x, board_o, isX):
        moves = []
        occupied = board_x | board_o
        for c in range(7):
            bottom_bit = 1 << (c * 7)
            if isX:
                if board_x & bottom_bit: moves.append((bottom_bit, True))
            else:
                if board_o & bottom_bit: moves.append((bottom_bit, True))
            top_bit = 1 << (c * 7 + 5)
            if not (occupied & top_bit):
                column_mask = 0b111111 << (c * 7)
                empty_in_col = (~occupied) & column_mask
                lowest_empty = empty_in_col & -empty_in_col
                moves.append((lowest_empty, False))
        return moves
    
    def apply_bit_pop(self, board_x, board_o, move_bit):
        col_idx = 0
        temp_bit = move_bit
        while temp_bit > 0b111111:
            temp_bit >>= 7
            col_idx += 1
        col_mask = 0b111111 << (col_idx * 7)
        bits_above_mask = (col_mask ^ move_bit) & (-(move_bit << 1))
        new_x = (board_x & ~col_mask) | ((board_x & bits_above_mask) >> 1)
        new_o = (board_o & ~col_mask) | ((board_o & bits_above_mask) >> 1)
        return new_x, new_o

    def check_bit_win(self, bitboard):
        m = bitboard & (bitboard >> 7)
        if m & (m >> 14): return True
        m = bitboard & (bitboard >> 1)
        if m & (m >> 2): return True
        m = bitboard & (bitboard >> 6)
        if m & (m >> 12): return True
        m = bitboard & (bitboard >> 8)
        if m & (m >> 16): return True
        return False

In [65]:
def mcts_base(rootBoard, isX: bool, iterations=10000, constant=1.41):
    
    # variavel responsavel por guardar o estado atual
    rootNode = MCTSNode(rootBoard, None, None, constant, isX)

    for _ in range(iterations):                   
        node = rootNode

        # Seleção
        while node.is_fully_expanded() and not node.is_terminal():
            node = node.select()

        # Expansão
        if not node.is_terminal():
            node = node.expand()
        
        # Simulação
        result = node.rollout()

        # BackPropagation
        node.update(result)

    best_move_node = max(rootNode.children, key=lambda c: c.visits)
    
    return best_move_node.move, best_move_node.move

In [ ]:
def mcts_search(rootBoard, isX: bool, iterations=10000):


    # 50% de chance de seguir o comportamento padrão do MCTS ou fazer um sorteio de probabilidades
    deterministic = True
    if random.randint(0, 1) == 1:
        deterministic = False
        
    # A cada jogada obtem uma nova constante
    constant = 0.91 + random.random()
    
    rootNode = MCTSNode(rootBoard, None, None, constant, isX)
    
    for _ in range(iterations):                   
        node = rootNode
        while node.is_fully_expanded() and not node.is_terminal():
            node = node.select()
        if not node.is_terminal():
            node = node.expand()
        result = node.rollout()
        node.update(result)
        
    best_move_node = max(rootNode.children, key=lambda c: c.visits)
    
    if deterministic:
        # Aqui o algoritmo segue o comportamento normal
        return best_move_node.move, best_move_node.move
    else:
        # Cria uma nova lista com o número de visitas que cada filho teve
        visit_counts = [child.visits for child in rootNode.children]

        # Guarda o total de visitas dos filhos
        total_visits = sum(visit_counts)

        # Sorteia um número no intervalo [0, total_visits[
        draw = random.random() * total_visits

        # Index para varrer a lista 
        id = -1

        
        while draw > 0:
            id += 1
            draw -= visit_counts[id]
        return rootNode.children[id].move, best_move_node.move

In [ ]:
class MCTSPlayer(Player):
    def turn(self, board, printer=True):
        if printer:
            print(f"{self}'s turn (MCTS pensando...)")
        
        move = mcts_base(board, self.isX, 10000)
        
        board.playMove(str(self), move[0]) 
        return move[1]

In [ ]:
print("--- Humano Vs MCTS ---")
if random.randint(0, 1) == 0:
    playerX = HumanPlayer(isX=True)
    playerO = MCTSPlayer(isX=False)
else:
    playerX = MCTSPlayer(isX=True)
    playerO = HumanPlayer(isX=False)

board = Board()
print(board)
while board.state == ' ':
    playerX.turn(board)
    print(board)
    if board.state != ' ':
        break
    playerO.turn(board)
    print(board)
print(board.state)

In [ ]:
print("--- Iniciando: MCTS Vs MCTS ---")

playerX = MCTSPlayer(isX=True)
playerO = MCTSPlayer(isX=False)

board = Board()
print(board)

while board.state == ' ':
    playerX.turn(board)
    print(board)
    
    if board.state != ' ':
        break
        
    playerO.turn(board)
    print(board)

print(board.state)

In [70]:
import time
import copy
import csv
import random
import os

def gerar_dataset_7h():
    print("--- Iniciando geração de Dataset com Aberturas Aleatórias (7 Horas) ---")
    
    # Define o local do arquivo
    caminho_arquivo = '../data/raw/MCTS_Random_Opening_data3.csv'
    
    # Garante que a pasta 'data/raw' existe. Se não existir, o Python cria agora!
    os.makedirs(os.path.dirname(caminho_arquivo), exist_ok=True)
    
    # Inicializa os jogadores
    playerX = MCTSPlayer(True) 
    playerO = MCTSPlayer(False)
    
    # 1. Cria o arquivo e escreve apenas o Cabeçalho
    header = ["isX"] + [f"c{i}" for i in range(42)] + ["target"]
    with open(caminho_arquivo, 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        
    start_time = time.time()
    seven_hours = 7 * 60 * 60 # 7 horas em segundos
    games_played = 0
    
    # 2. Inicia o cronômetro
    while time.time() - start_time < seven_hours:
        board = Board()
        game_data = [] 
        
        while board.state == ' ':
            storedBoard = copy.deepcopy(board.board)
            
            # --- ABERTURA ALEATÓRIA PARA O X ---
            if board.empty > 38:
                moves = playerX.getPossibleMoves(board)
                move = random.choice(moves)
                board.playMove(str(playerX), move)
            else:
                move = playerX.turn(board, False) # MCTS assume o controle
            # -----------------------------------
            
            newRow = [1]
            for r in range(6): 
                for c in range(7):
                    if storedBoard[r][c] == "X":
                        newRow.append(1)
                    elif storedBoard[r][c] == "O":
                        newRow.append(-1)
                    else:
                        newRow.append(0)
            newRow.append(move)
            game_data.append(newRow)
            
            if board.state != ' ':
                break
                
            storedBoard = copy.deepcopy(board.board)
            
            # --- ABERTURA ALEATÓRIA PARA O O ---
            if board.empty > 38:
                moves = playerO.getPossibleMoves(board)
                move = random.choice(moves)
                board.playMove(str(playerO), move)
            else:
                move = playerO.turn(board, False) # MCTS assume o controle
            # -----------------------------------
            
            newRow = [0]
            for r in range(6):
                for c in range(7):
                    if storedBoard[r][c] == "O":
                        newRow.append(-1)
                    elif storedBoard[r][c] == "X":
                        newRow.append(1)
                    else:
                        newRow.append(0)
            newRow.append(move)
            game_data.append(newRow)
            
        # 3. Salva o jogo atual no arquivo sem apagar os anteriores ('a' de append)
        with open(caminho_arquivo, 'a', newline='') as file:
            writer = csv.writer(file)
            writer.writerows(game_data)
            
        games_played += 1
        elapsed_time = time.time() - start_time
        print(f"Jogo {games_played} concluído. Tempo rodando: {elapsed_time/3600:.2f}/7.00 horas")

    print(f"\nGeração de dataset concluída com sucesso! Arquivo salvo em: {caminho_arquivo}")

# Chama a função para rodar
gerar_dataset_7h()

--- Iniciando geração de Dataset com Aberturas Aleatórias (7 Horas) ---
Jogo 1 concluído. Tempo rodando: 0.01/7.00 horas
Jogo 2 concluído. Tempo rodando: 0.02/7.00 horas
Jogo 3 concluído. Tempo rodando: 0.04/7.00 horas
Jogo 4 concluído. Tempo rodando: 0.04/7.00 horas
Jogo 5 concluído. Tempo rodando: 0.06/7.00 horas
Jogo 6 concluído. Tempo rodando: 0.06/7.00 horas
Jogo 7 concluído. Tempo rodando: 0.07/7.00 horas
Jogo 8 concluído. Tempo rodando: 0.07/7.00 horas
Jogo 9 concluído. Tempo rodando: 0.08/7.00 horas


KeyboardInterrupt: 

In [23]:
# 3) implementar e aplicar ID3
import numpy as np #importa numpy para calculos precisos matematicos

#3.1) Calculo de entropia (pagina 30 - slide 7)
def entropia(y):
    nome_classes, contagem = np.unique(y,return_counts=True) #funcao unique retorna o nome e a quantidade de cada classe
    # Iris-setosa ou Iris-versicolor ou Iris-virginica
    prob = contagem/len(y) #prob é a quantidade que uma class aparece / tamanho do dataset
    return -np.sum(prob*np.log2(prob)) #Entropia: soma de -p * log2(p) (mensura a incerteza em cima de uma alguma varivael X)

#3.2) Calculo de ganho de informaçao (pagina 31 - slide 7)
def ganho_informacao(df,atributo,y):
    entropia_total = entropia(y) #calcula a entropia H(c)
    baldes,contagens = np.unique(df[atributo],return_counts=True) #retorna os valores e quantidade de cada 'balde' dentro de uma feature em X
    entropia_condiconal = 0 #inicial em 0

    for balde in range(len(baldes)): # H(C|Atributo) ou seja a prob de Y sabendo de um atributo (de X) em especifico
        subset = df[df[atributo] == baldes[balde]] #define um subset de um atributo em X por cada balde (valores da discretizaçao)
        prob_subgrupo = contagens[balde]/len(df) #calcula a prob de um atributo em especifico aparecer no dataset
        entropia_condiconal += prob_subgrupo* entropia(subset[y]) #soma das prob de cada subgrupo * a entopia do total do subgrupo
    return entropia_total - entropia_condiconal #ganho de informaçao (slide 33 - Ganho(A) = I(C;A) = H(C)-H(C|A))
    
#3.3) implememtaçao do ID3 (implementado a partir do pseudocodigo da pagina 35 - slide 7)
def my_ID3(Examples,X,y):
    if len(np.unique(y)) == 1: #se todos os exemplos tem o mesmo valor de target retorna a root com esse valor - caso base
        return np.unique(y)[0]
    if X.empty: #se nao tem variavel independente retorna a moda de y - case base quando ja reparou todos as features
        return y.mode()[0]
    
    ganhos = [] #cria lista de ganhos
    for var_ind in X.columns: #calcula o ganho de informação de casa variavel independente 
        valor_ganho = ganho_informacao(Examples,var_ind,y.name) 
        ganhos.append(valor_ganho) #add ganho i na lista de ganhos

    indice_melhor = np.argmax(ganhos) #acha o indce da lista do maior ganho
    A = X.columns[indice_melhor] #define A como o maior dos ganhos
    #A é definido como o melhor classificador do dataset
    tree = {A: {}} #cria a root da arvore (com inmpletaçao de dicionario)
    #pega o melhor atributo A, e adiciona os baldes como chave externa
    #exemplo, se A = 'petallenhth', tree = {'petallenght" = {}}
    #tree['petallength']['Pequeno'] = 'Iris-setosa' #folha
    #tree['petallength']['Grande'] = {'sepalwidth': {}} #outro nó (sub-árvore)
    for valor in Examples[A].unique(): #itera sobre os valores possives dado a feature escolhida A
        subset = Examples[Examples[A] == valor] #faz com que o set de exemplos analisados seja Examples(vi)
        if subset.empty: #se esse subset for vazio
            tree[A][valor] = y.mode()[0] #escolhe a moda de y
        else: #cria nova  lista de var inde tirando antigo A
            novos_X = X.drop(A, axis=1) #remove o A que ja foi escolhido
            novo_y = subset[y.name]
            tree[A][valor] = my_ID3(subset,novos_X,novo_y) #chama recursivamente a funçao para os novos valores
    return tree

#3.4) define classifcador com o ID3
def classificador(exemplo,arvore):
    if not isinstance(arvore,dict): #se a arvore nao der um dict, chegamos em uma folha
        return arvore
    
    atributo = list(arvore.keys())[0] #pega o atributo que está no no da arvore
    valor_exemplo = exemplo[atributo] #pega o valor q o exemplo tem paar esse atributo

    if valor_exemplo in arvore[atributo]: #segue pelo ramo correspondente
        sub_arvore = arvore[atributo][valor_exemplo]
        return classificador(exemplo, sub_arvore)
    else: #caso o valor nao existe - !!!(talvez posso duar aqui para retornar a moda)!!!
        return "Classe Desconhecida"

&nbsp;Para o desenvolvimento do dataset do jogo **PopOut** em princípio iremos utilizar o **MCTS** com os parametros ???, deixamos esse algoritmo jogar o jogo por algumas horas e obtemos o seguinte dataset:

In [24]:
import pandas as pd

df = pd.read_csv('../data/raw/MCTS_Random_Opening_data.csv')
df.head(5)

,isX,c0,c1,c2,c3,c4,c5,c6,c7,c8,...,c33,c34,c35,c36,c37,c38,c39,c40,c41,target
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"(1, 5)"
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,"(1, 7)"
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,-1,"(2, 5)"
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,-1,"(1, 2)"
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,-1,0,0,1,0,-1,"(3, 5)"


&nbsp;Esse dataset tem 44 colunas, sendo a primeira coluna um identificador do jogador, as seguintes 42 colunas se referem aos campos do tabuleiro e seu conteudo, com **1** representando que aquele espaço esta sendo ocupado pelo jogador **MAX** (**'X'**), **-1** o jogador **MIN** (**'O'**) e 0 um espaço ainda não ocupado, a ultima coluna do dataset contém a informação do melhor movimento a ser feito, que foi decidido pelo **MCTS**. Para treinar a nossa árvore de decisão usando o algoritmo **ID3** nós definimos como variaveis independentes (features) as 43 primeiras colunas e a variavel dependente (target) representada pela 44ª coluna, definimos então nosso $X$ e $y$:

In [35]:
import ast

X = df.drop('target', axis=1)
y = df['target'].apply(ast.literal_eval)

tree_MCTS_RO = my_ID3(df,X,y)

In [ ]:
#from pprint import pprint
#pprint(tree_MCTS_RO, indent=4)

df_blind = df.drop('target',axis=1)

acertos = 0
lista_erros = []

for i in range(len(df)):
    exemplo = df_blind.iloc[i].to_dict()
    real    = df.iloc[i]['target']
    predicao = classificador(exemplo,tree_MCTS_RO)
    if real == predicao:
        acertos = acertos+1
    else:
        lista_erros.append(i+1)
        #print(f"predicao = {predicao} real = {real}")

print(f"Tamanho = {len(df)}\nAcertos = {acertos}")
print(f"acuracia = {acertos/len(df)}")

In [37]:
X = df.drop('target', axis=1)
y = df['target'].apply(ast.literal_eval)

split_index = int(len(df)*0.7)
df_treino = df.iloc[:split_index]
df_teste  = df.iloc[split_index:]

print(f"tamanho total  = {len(df)}")
print(f"tamanho treino = {len(df_treino)}")
print(f"tamanho teste  = {len(df_teste)}")

tree_MCTS_RO_treino = my_ID3(df_treino,X,y)

tamanho total  = 13708
tamanho treino = 9595
tamanho teste  = 4113


In [ ]:
acertos_treino = 0
lista_erros = []

df_teste_blind = df_teste.drop('target', axis=1)

for i in range(len(df_teste)):
    exemplo = df_teste_blind.iloc[i].to_dict()
    real = df_teste.iloc[i]['target']
    predicao = classificador(exemplo,tree_MCTS_RO_treino)
    if real == predicao:
        acertos_treino = acertos_treino+1
    else:
        lista_erros.append(i+1)
        #print(f"predicao = {predicao} real = {real}")
print(f"Tamanho = {len(df_teste)}\nAcertos = {acertos_treino}")
print(f"acuracia = {acertos_treino/len(df_teste)}")
    

Tamanho = 4113
Acertos = 1044
acuracia = 0.2538293216630197


**DataSet $Simão$**

In [50]:
import pandas as pd

dfSimao = pd.read_csv('../MCTStraining.csv')
dfSimao.head(5)

,isX,c0,c1,c2,c3,c4,c5,c6,c7,c8,...,c33,c34,c35,c36,c37,c38,c39,c40,c41,target
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"(1, 4)"
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,"(2, 4)"
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,"(1, 5)"
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,0,0,"(1, 3)"
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,-1,1,1,0,0,"(1, 6)"


In [52]:
import ast

Xsimao = dfSimao.drop('target', axis=1)
ysimao = dfSimao['target'].apply(ast.literal_eval)

tree_simao = my_ID3(dfSimao,Xsimao,ysimao)

In [ ]:
#from pprint import pprint
#pprint(tree_MCTS_RO, indent=4)

df_blind_sim = df.drop('target',axis=1)

acertos = 0
lista_erros = []

for i in range(len(df)):
    exemplo = df_blind_sim.iloc[i].to_dict()
    real    = df.iloc[i]['target']
    predicao = classificador(exemplo,tree_MCTS_RO)
    if real == predicao:
        acertos = acertos+1
    else:
        lista_erros.append(i+1)
        #print(f"predicao = {predicao} real = {real}")

print(f"Tamanho = {len(df)}\nAcertos = {acertos}")
print(f"acuracia = {acertos/len(df)}")

In [69]:
import time
import copy
import csv
import random
import os
import multiprocessing

# 1. Isolamos a lógica principal em uma função que recebe um "id" (ex: 'A' ou 'B')
def trabalhador_dataset(id_processo, horas_limite):
    print(f"--- [Processo {id_processo}] Iniciando geração (Aberturas Aleatórias) ---")
    
    # Define o local do arquivo específico para este processo
    caminho_arquivo = f'../data/raw/MCTS_Random_Opening_data2_{id_processo}.csv'
    
    os.makedirs(os.path.dirname(caminho_arquivo), exist_ok=True)
    
    playerX = MCTSPlayer(True) 
    playerO = MCTSPlayer(False)
    
    header = ["isX"] + [f"c{i}" for i in range(42)] + ["target"]
    with open(caminho_arquivo, 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        
    start_time = time.time()
    tempo_limite_segundos = horas_limite * 60 * 60
    games_played = 0
    
    while time.time() - start_time < tempo_limite_segundos:
        board = Board()
        game_data = [] 
        
        while board.state == ' ':
            storedBoard = copy.deepcopy(board.board)
            
            # --- ABERTURA ALEATÓRIA PARA O X ---
            if board.empty > 38:
                moves = playerX.getPossibleMoves(board)
                move = random.choice(moves)
                board.playMove(str(playerX), move)
            else:
                move = playerX.turn(board, False)
            
            newRow = [1]
            for r in range(6): 
                for c in range(7):
                    if storedBoard[r][c] == "X":
                        newRow.append(1)
                    elif storedBoard[r][c] == "O":
                        newRow.append(-1)
                    else:
                        newRow.append(0)
            newRow.append(move)
            game_data.append(newRow)
            
            if board.state != ' ':
                break
                
            storedBoard = copy.deepcopy(board.board)
            
            # --- ABERTURA ALEATÓRIA PARA O O ---
            if board.empty > 38:
                moves = playerO.getPossibleMoves(board)
                move = random.choice(moves)
                board.playMove(str(playerO), move)
            else:
                move = playerO.turn(board, False)
            
            newRow = [0]
            for r in range(6):
                for c in range(7):
                    if storedBoard[r][c] == "O":
                        newRow.append(-1)
                    elif storedBoard[r][c] == "X":
                        newRow.append(1)
                    else:
                        newRow.append(0)
            newRow.append(move)
            game_data.append(newRow)
            
        with open(caminho_arquivo, 'a', newline='') as file:
            writer = csv.writer(file)
            writer.writerows(game_data)
            
        games_played += 1
        elapsed_time = time.time() - start_time
        # O print agora mostra qual processo terminou o jogo
        print(f"[Processo {id_processo}] Jogo {games_played} concluído. Tempo: {elapsed_time/3600:.2f}/{horas_limite:.2f}h")

    print(f"\n[Processo {id_processo}] Concluído! Arquivo salvo em: {caminho_arquivo}")


# 2. A função principal que gerencia o paralelismo
def gerar_datasets_paralelos(horas=7):
    processo_A = multiprocessing.Process(target=trabalhador_dataset, args=('A', horas))
    processo_B = multiprocessing.Process(target=trabalhador_dataset, args=('B', horas))
    
    processo_A.start()
    processo_B.start()
    
    processo_A.join()
    processo_B.join()
    
    print("\nTODOS OS PROCESSOS FORAM CONCLUÍDOS COM SUCESSO!")

if __name__ == '__main__':
    # 1. Importe a biblioteca caso não esteja no topo
    import multiprocessing 
    
    # 2. FORCE O LINUX A USAR O METODO FORK
    multiprocessing.set_start_method('fork', force=True) 
    
    # 3. Chama a função principal
    gerar_datasets_paralelos(horas=7)

--- [Processo A] Iniciando geração (Aberturas Aleatórias) ---
--- [Processo B] Iniciando geração (Aberturas Aleatórias) ---
[Processo B] Jogo 1 concluído. Tempo: 0.01/7.00h
[Processo A] Jogo 1 concluído. Tempo: 0.01/7.00h
[Processo B] Jogo 2 concluído. Tempo: 0.02/7.00h
[Processo A] Jogo 2 concluído. Tempo: 0.02/7.00h
[Processo A] Jogo 3 concluído. Tempo: 0.03/7.00h
[Processo B] Jogo 3 concluído. Tempo: 0.03/7.00h
[Processo A] Jogo 4 concluído. Tempo: 0.04/7.00h
[Processo B] Jogo 4 concluído. Tempo: 0.04/7.00h
[Processo A] Jogo 5 concluído. Tempo: 0.04/7.00h
[Processo B] Jogo 5 concluído. Tempo: 0.05/7.00h
[Processo A] Jogo 6 concluído. Tempo: 0.05/7.00h
[Processo A] Jogo 7 concluído. Tempo: 0.05/7.00h
[Processo B] Jogo 6 concluído. Tempo: 0.05/7.00h
[Processo A] Jogo 8 concluído. Tempo: 0.05/7.00h
[Processo B] Jogo 7 concluído. Tempo: 0.06/7.00h
[Processo A] Jogo 9 concluído. Tempo: 0.06/7.00h
[Processo B] Jogo 8 concluído. Tempo: 0.07/7.00h
[Processo A] Jogo 10 concluído. Tempo: 0.07

Process Process-6:
Process Process-5:
Traceback (most recent call last):


KeyboardInterrupt: 

Traceback (most recent call last):
  File "/home/rafael/anaconda3/envs/ia/lib/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/home/rafael/anaconda3/envs/ia/lib/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/home/rafael/anaconda3/envs/ia/lib/python3.14/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/rafael/anaconda3/envs/ia/lib/python3.14/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_4523/3060259652.py", line 67, in trabalhador_dataset
    move = playerO.turn(board, False)
  File "/tmp/ipykernel_4523/3060259652.py", line 67, in trabalhador_dataset
    move = playerO.turn(board, False)
  File "/tmp/ipykernel_4523/3889599675.py", line 6, in turn
    move = mcts_sea

  File "/tmp/ipykernel_4523/3889599675.py", line 6, in turn
    move = mcts_search(board, self.isX, 10000)
  File "/tmp/ipykernel_4523/796610146.py", line 19, in mcts_search
    node = node.select()
  File "/tmp/ipykernel_4523/796610146.py", line 19, in mcts_search
    node = node.select()
  File "/tmp/ipykernel_4523/3547639504.py", line 19, in select
    return max(self.children, key=lambda c: (c.wins / c.visits) + self.constant * np.sqrt(np.log(self.visits) / c.visits))
  File "/tmp/ipykernel_4523/3547639504.py", line 19, in select
    return max(self.children, key=lambda c: (c.wins / c.visits) + self.constant * np.sqrt(np.log(self.visits) / c.visits))
KeyboardInterrupt
  File "/tmp/ipykernel_4523/3547639504.py", line 19, in <lambda>
    return max(self.children, key=lambda c: (c.wins / c.visits) + self.constant * np.sqrt(np.log(self.visits) / c.visits))
    
KeyboardInterrupt
